[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_Math/Concentration/Concentration_Inequalities.ipynb)


Content for and by IEEE Signal Processing Society. (Raul Valle & Contributors)

# Concentration Inequalities & Learning Theory

> ⚠️ **Draft — pending instructor review.** Simulations execute and corroborate every bound, but execution cannot verify proofs. Review before teaching; remove this banner after.

Why do sample averages of *bounded* things behave so much better than [Chebyshev](../Analysis/Random_Variables.ipynb) promises? Exponentially better — and that exponential is the engine under every generalization bound in machine learning. Four sessions: sub-Gaussian variables → Hoeffding → McDiarmid → Rademacher complexity, ending with an honest generalization bound computed for a model you trained.

## 1. Pre-requisites

[Random Variables](../Analysis/Random_Variables.ipynb) (Markov/Chebyshev — about to be embarrassed), [Independence](../Analysis/Independence.ipynb) (i.i.d., LLN).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 4 — *Sub-Gaussian Variables & the Chernoff Trick* (~40 min)
**Goal:** upgrade Markov from polynomial to exponential by applying it to e^{λX}.
**Builds on:** [Random Variables](../Analysis/Random_Variables.ipynb). &nbsp; **Feeds into:** Session 2 (Hoeffding).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: Sub-Gaussian Variables & the Chernoff Trick</b></summary>

**Timing (~40 min).** 8 min why Chebyshev is weak · 12 min the Chernoff trick · 10 min sub-Gaussianity and Hoeffding's lemma · 10 min the demo.

**Board first — diagnose Chebyshev's weakness precisely.** It uses only the mean and variance, so it must hold for *every* distribution with that variance, including the worst one — a two-point distribution that puts all its mass at the extremes. Being valid for the worst case makes it loose for every reasonable case. Ask the room what extra information could tighten it; the answer is the whole moment generating function, and that is exactly what Chernoff exploits.

**The Chernoff trick is one line, and it is the engine of the entire workshop.** Apply Markov not to $X$ but to $e^{\lambda X}$. The exponential amplifies the tail, so controlling $E[e^{\lambda X}]$ controls the tail exponentially rather than polynomially — then optimise over $\lambda$ to get the best bound available. Do the three lines on the board; students who see the optimisation step ($\lambda = t/\sigma^2$) understand where the $t^2$ in the exponent comes from.

**Sub-Gaussianity, stated as a comparison.** A variable is sub-Gaussian if its MGF *never beats a Gaussian's*. That is the whole condition, and it immediately yields $P(X-\mu \ge t) \le e^{-t^2/2\sigma^2}$ — Gaussian-grade tails without assuming a Gaussian.

**Hoeffding's lemma is the surprise worth dwelling on.** *Boundedness alone* buys sub-Gaussianity with proxy $(b-a)^2/4$. No distributional assumption whatsoever — not symmetry, not unimodality, nothing. If your random variable lives in a bounded interval, it has Gaussian tails. Ask why that is plausible: a bounded variable simply has no room for a heavy tail, so the only question is the constant. This is the single most reusable fact in the workshop, because losses in machine learning are almost always bounded.

**Frame the demo as "both bounds are valid; only one has the right shape."** On the log axis, Chebyshev's curve is a straight-ish polynomial decay while the sub-Gaussian one bends downward. Point out that neither is *tight* — both sit well above the empirical curve — and that the useful comparison is the slope, not the offset. A bound with the right shape becomes useful as $n$ grows; a bound with the wrong shape never does.

**Ask the room.** "The sub-Gaussian bound is still loose. So why prefer it?" Because looseness by a constant factor is survivable, and looseness by a *functional form* is not. Session 2 makes that concrete by converting the exponent into a sample-size requirement.
</details>

## 2. The One Trick Underneath Everything

💡 **Intuition.** Markov's inequality is weak because it only uses the mean. The **Chernoff trick**: apply Markov not to $X$ but to $e^{\lambda X}$ — the exponential amplifies tail behavior, and optimizing over $\lambda$ squeezes out an *exponentially* small bound wherever the moment generating function is controlled. A variable is **sub-Gaussian** with proxy $\sigma^2$ if $E[e^{\lambda(X-\mu)}] \le e^{\lambda^2\sigma^2/2}$ — its MGF never beats a Gaussian's — and then
$$P(X - \mu \ge t) \le e^{-t^2 / 2\sigma^2}.$$
*Proof.* $P(X - \mu \ge t) = P(e^{\lambda(X-\mu)} \ge e^{\lambda t}) \le e^{-\lambda t} E[e^{\lambda(X-\mu)}] \le e^{-\lambda t + \lambda^2\sigma^2/2}$; minimize at $\lambda = t/\sigma^2$. $\blacksquare$

**Hoeffding's lemma** (stated; proof is a clean convexity exercise): any variable confined to $[a, b]$ is sub-Gaussian with $\sigma^2 = (b-a)^2/4$ — *boundedness alone buys Gaussian-grade tails*, no distributional assumptions.

In [2]:
# Chebyshev vs sub-Gaussian tail for a bounded variable (uniform on [-1, 1])
t_grid = np.linspace(0.3, 0.95, 20)
X = rng.uniform(-1, 1, 2_000_000)
empirical = [(X > t).mean() for t in t_grid]
cheb = np.minimum(1, (1/3) / t_grid**2)              # Var = 1/3
subg = np.exp(-t_grid**2 / (2 * 1.0))                # (b-a)²/4 = 1

plt.figure(figsize=(7.5, 3))
plt.semilogy(t_grid, empirical, "o-", label="empirical  P(X > t)")
plt.semilogy(t_grid, cheb, "--", label="Chebyshev bound (polynomial)")
plt.semilogy(t_grid, subg, "--", label="sub-Gaussian bound (exponential)")
plt.legend(); plt.xlabel("t"); plt.grid(True, alpha=0.3)
plt.title("both bounds are valid; only one has the right SHAPE")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2978551/604148098.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Three curves on a log axis: the empirical tail, Chebyshev's bound, and the sub-Gaussian bound. **Both bounds are valid** — neither is ever crossed — but they have different *shapes*, and the title is right that shape is what matters.

Chebyshev decays polynomially, like $1/t^2$; the sub-Gaussian bound decays like $e^{-t^2/2}$. On a log-linear plot that is a slowly-flattening curve against one that keeps bending downward. As $t$ grows the gap between them widens without limit.

**Why Chebyshev has to be weak.** It uses only the mean and variance, so it must be valid for *every* distribution with that variance — including the worst one, which puts all its mass at two extreme points. A bound that survives the adversarial case is necessarily loose for a well-behaved one. Nothing is wrong with it; it is simply answering a harder question than we asked.

**And the Chernoff trick is what buys the improvement.** Apply Markov's inequality not to $X$ but to $e^{\lambda X}$. The exponential amplifies the tail, so controlling the moment generating function controls the tail *exponentially*; optimising over $\lambda$ (at $\lambda = t/\sigma^2$) extracts the best available bound and produces the $t^2$ in the exponent. That single move — Markov applied to an exponential — generates every result in this workshop.

**The most reusable fact here is Hoeffding's lemma, and it is genuinely surprising.** Any variable confined to $[a,b]$ is sub-Gaussian with proxy $(b-a)^2/4$. **Boundedness alone buys Gaussian-grade tails** — no symmetry assumption, no unimodality, no distributional form at all. A bounded variable has nowhere to put a heavy tail, so the only remaining question is the constant.

That is why this workshop matters to machine learning: losses are almost always bounded (a 0–1 loss trivially, most others by construction), so Hoeffding applies to empirical risks without anyone having to argue about the data distribution.

**One honest observation about the plot.** Neither bound is tight — both sit visibly above the empirical curve everywhere, and the sub-Gaussian one is loose by a substantial factor. That is expected and it is not the point. The useful comparison is the *slope*: a bound with the right functional form becomes progressively more useful as the deviation or the sample size grows, while a bound with the wrong form never catches up regardless. Session 2 converts that exponent into a concrete sample-size requirement.

---
### 🕐 Session 2 of 4 — *Hoeffding & the Price of Confidence* (~35 min)
**Goal:** exponential concentration for sample means; how many samples for ±1% at 99.9%?
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (McDiarmid).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: Hoeffding & the Price of Confidence</b></summary>

**Timing (~35 min).** 8 min why proxies add · 10 min inverting the bound · 12 min the demo including its vacuous row · 5 min the sample-size answer.

**Board first — why sums work.** Independence factorises the MGF, so sub-Gaussian *proxies add*. That single line is why Hoeffding follows from Session 1 with no new ideas: a sum of $n$ bounded variables is sub-Gaussian with proxy $\sum(b_i-a_i)^2/4$, and dividing by $n$ to get the mean produces the $n$ in the exponent. Point out that [Independence](../Analysis/Independence.ipynb)'s $E[XY] = E[X]E[Y]$ is doing all the work.

**The inverted form is the practically useful one.** $t = \sqrt{\ln(2/\delta)/2n}$. Read the asymmetry aloud: **confidence is exponentially cheap, precision is quadratically expensive.** Going from 95% to 99.9% confidence multiplies the requirement by a modest factor (the $\ln$), while halving the error bar quadruples the sample size. Ask which knob a practitioner should turn first — almost always confidence, because it is nearly free.

**Do the sample-size calculation as the session's deliverable.** 38,005 samples for ±1% at 99.9% confidence, *regardless of the distribution* provided it is bounded. That is a number a student can take to an experiment design meeting, and its distribution-freeness is the remarkable part.

**Handle the first row of the demo carefully — it is vacuous and you should say so.** At $t = 0.02$ the bound is **1.3406**, which is greater than 1 and therefore says nothing at all: probabilities are always at most 1, so the bound is trivially true and carries zero information. Ask the room to spot that before you point it out. It is a good habit — a bound above 1, or a variance bound below 0, is a signal that the regime is outside where the inequality is useful.

**Then the other two rows, which are informative and loose.** 0.0734 against 0.4038 (5.5× loose) and 0.0078 against 0.0546 (7× loose). Both hold, both are honest, neither is tight. Emphasise that Hoeffding does not use the *variance* — it only uses the range — so for a fair coin, whose variance is much smaller than its range suggests, it is systematically conservative. Bernstein's inequality tightens exactly this by including the variance, which is the natural follow-up question if a student asks.

**Ask the room.** "Should I use this bound to decide how many samples to collect?" Yes, and expect to over-collect. Hoeffding is a *guarantee*, and guarantees are priced for the worst case. If sampling is cheap, use it directly; if each sample costs a thousand dollars, invest in a tighter bound (Bernstein, empirical Bernstein) or accept an assumption about the distribution.
</details>

## 3. Hoeffding's Inequality

Independent $X_i \in [a_i, b_i]$: sums of sub-Gaussians are sub-Gaussian with proxies *adding* (independence factorizes the MGF — [Independence](../Analysis/Independence.ipynb)'s $E[XY]=E[X]E[Y]$ doing the work), so
$$P\big(|\bar{X}_n - \mu| \ge t\big) \le 2\exp\Big(\frac{-2n^2t^2}{\sum_i (b_i - a_i)^2}\Big).$$

💡 **Intuition.** Chebyshev said deviation probability falls like $1/nt^2$; Hoeffding says like $e^{-2nt^2}$. Inverted: confidence $1-\delta$ costs $t = \sqrt{\ln(2/\delta) / 2n}$ — **confidence is exponentially cheap**, precision is quadratically expensive. That square root is the same $\sqrt{T}$ shape as [regret bounds](../../Intro_Time_Series/Online_Learning_and_Regret.ipynb) — not a coincidence, the same Chernoff machinery lives under both.

In [3]:
# The bound, audited: coin flips, n = 500
n_samp, trials = 500, 200_000
means = (rng.random((trials, n_samp)) < 0.5).mean(1)
for t in [0.02, 0.04, 0.06]:
    emp = (np.abs(means - 0.5) >= t).mean()
    bound = 2*np.exp(-2*n_samp*t**2)
    print(f"t={t}: empirical {emp:.4f} ≤ Hoeffding {bound:.4f}  ✓" )
n_needed = int(np.ceil(np.log(2/0.001) / (2*0.01**2)))
print(f"\nsamples for ±1% at 99.9% confidence: {n_needed:,} — REGARDLESS of the distribution (if bounded)")

t=0.02: empirical 0.3962 ≤ Hoeffding 1.3406  ✓
t=0.04: empirical 0.0734 ≤ Hoeffding 0.4038  ✓
t=0.06: empirical 0.0078 ≤ Hoeffding 0.0546  ✓

samples for ±1% at 99.9% confidence: 38,005 — REGARDLESS of the distribution (if bounded)


**What just happened.** The bound holds in every row — but read the first one carefully, because it is **vacuous**:

| $t$ | empirical | Hoeffding | verdict |
|---|---|---|---|
| 0.02 | 0.3962 | **1.3406** | says nothing — a probability cannot exceed 1 |
| 0.04 | 0.0734 | 0.4038 | valid, 5.5× loose |
| 0.06 | 0.0078 | 0.0546 | valid, 7.0× loose |

A bound of 1.3406 on a probability is trivially true and carries **zero information**. It is not wrong; it is simply outside the regime where the inequality is useful, because $2e^{-2nt^2}$ only drops below 1 once $nt^2$ is large enough. Spotting a vacuous bound is a genuinely useful habit — like a variance bound that comes out negative, it tells you the deviation you asked about is too small for the sample size you have.

**The other two rows are honest and loose, and the reason is structural.** Hoeffding uses only the *range* of each variable, never its variance. A fair coin has range 1 but variance only 0.25, so a bound calibrated for the worst distribution on $[0,1]$ — one placing all mass at the endpoints — is necessarily conservative here. Bernstein's inequality tightens exactly this by using the variance as well, and is the natural next tool when a factor of 7 matters.

**Now the number that makes the session worth teaching.** Inverting the bound gives
$$t = \sqrt{\frac{\ln(2/\delta)}{2n}},$$
and solving for ±1% at 99.9% confidence needs **38,005 samples** — *regardless of the distribution*, provided it is bounded. No normality assumption, no variance estimate, no asymptotics. That is a number you can take into an experiment-design meeting.

**And read the asymmetry in that formula, because it is the practical lesson.** $\delta$ appears inside a logarithm while $t$ appears squared. So **confidence is exponentially cheap and precision is quadratically expensive**: tightening 95% → 99.9% costs a modest factor, while halving the error bar *quadruples* the sample size. When a study is under-powered, buying confidence is almost always the cheaper move.

Note also that this is the same $\sqrt{\cdot/n}$ shape as the $\sqrt{T}$ in [regret bounds](../../Intro_Time_Series/Online_Learning_and_Regret.ipynb) and the $1/\sqrt{k}$ in [Kalman](../../Intro_Time_Series/Intro_AdFilt_KF.ipynb)'s uncertainty. Not a coincidence — the same Chernoff machinery underlies all three.

**One caveat on using this for planning.** Hoeffding is a *guarantee*, priced for the worst case, so following it means over-collecting. If samples are cheap, that is the right trade. If each one costs real money, it is worth reaching for a tighter bound or accepting a distributional assumption in exchange for a smaller $n$.

---
### 🕐 Session 3 of 4 — *McDiarmid: Beyond Sums* (~35 min)
**Goal:** concentration for ANY stable function of independent variables.
**Builds on:** Session 2. &nbsp; **Feeds into:** Session 4 (Rademacher & generalization).

---

<details>
<summary>🎓 <b>Teacher notes — Session 3: McDiarmid — Beyond Sums</b></summary>

**Timing (~35 min).** 10 min the generalisation from averages to stability · 8 min the bounded-differences condition · 10 min the demo · 7 min why this unlocks learning theory.

**Board first — name what was special about averages, then discard it.** Session 2 concentrated *sums*. Ask what property made that work. The instinct is "independence," but that is only half: the other half is that **no single term can move the average much** — changing one $X_i$ shifts $\bar X$ by at most $(b-a)/n$. McDiarmid's insight is that this *stability* is the real requirement, and averaging is merely one way to achieve it.

**State the bounded-differences condition carefully.** If changing any single input moves $f(X_1,\dots,X_n)$ by at most $c_i$, then $P(|f - Ef| \ge t) \le 2e^{-2t^2/\sum c_i^2}$ — the same Gaussian tail, for an arbitrary function. Emphasise that $f$ need not be an average, need not be linear, need not be continuous. It only has to be *insensitive to any one input*.

**Ask the room for a function that fails.** The maximum of $n$ variables: changing one input can move it by the full range, so $c_i$ is large and the bound is weak — correctly, since the max genuinely does not concentrate the way an average does. Having a non-example makes the condition feel like a real hypothesis rather than a formality.

**The demo's point is that the statistic is not an average.** Longest run of heads in 200 flips: no sum, no linearity, and yet mean 7.00 with standard deviation only 1.83 — tightly concentrated. Flipping a single coin changes the longest run by at most a small amount, so bounded differences applies and concentration follows. Be honest that the cell *demonstrates* concentration rather than computing an explicit $c_i$ and evaluating the bound; the code comment's "$c_i = \ldots$ small" is doing some hand-waving, and saying so is better than glossing it.

**Then the payoff sentence, which is why this session exists.** Empirical risk is an average of bounded losses, so swapping one training example moves it by at most $1/n$ — bounded differences with $c_i = 1/n$. But the object learning theory actually needs to control is $\sup_{f}|\text{train} - \text{true}|$, a *supremum over a model class*, which is emphatically not an average. McDiarmid still applies, because swapping one example moves that supremum by at most $1/n$ too. **That is the licence to talk about generalisation at all**, and Session 4 depends entirely on it.

**Mention the proof route briefly.** A martingale of conditional expectations, plus Hoeffding's lemma applied at each step — the Doob decomposition. Students who have done [Stochastic Processes II](../Stochastic_Processes/Stochastic_Processes_2.ipynb) will recognise it; those who have not can take it on trust, since the *statement* is what gets used.
</details>

## 4. Functions with Bounded Differences

💡 **Intuition.** Averages aren't special — **stability** is. If changing any one input moves $f(X_1..X_n)$ by at most $c_i$ (bounded differences), McDiarmid gives the same Gaussian tail: $P(|f - E f| \ge t) \le 2e^{-2t^2/\sum c_i^2}$. *(Proof route: a martingale of conditional expectations + Hoeffding's lemma per step — see [Stochastic Processes II](../Stochastic_Processes/Stochastic_Processes_2.ipynb).)* This is the license to talk about concentration of *empirical risks*, since swapping one training example moves an average of bounded losses by ≤ 1/n.

In [4]:
# McDiarmid in action on a non-average: longest run of heads in 200 flips (c_i = ... small)
def longest_run(flips):
    best = cur = 0
    for f in flips:
        cur = cur + 1 if f else 0
        best = max(best, cur)
    return best

runs = np.array([longest_run(rng.random(200) < 0.5) for _ in range(30000)])
print(f"longest-run statistic: mean {runs.mean():.2f}, std {runs.std():.2f}")
print(f"→ changing ONE flip changes the statistic by ≤ ~its neighborhood — tightly concentrated,")
print(f"  though it's nobody's average. Empirical P(|f−Ef| ≥ 4) = {(np.abs(runs-runs.mean())>=4).mean():.4f}")

longest-run statistic: mean 7.00, std 1.83
→ changing ONE flip changes the statistic by ≤ ~its neighborhood — tightly concentrated,
  though it's nobody's average. Empirical P(|f−Ef| ≥ 4) = 0.0231


**What just happened.** The longest run of heads in 200 flips has mean **7.00** and standard deviation only **1.83**, with $P(|f - Ef| \ge 4) = 0.023$. That is tight concentration — and the statistic is **nobody's average**.

That is the whole point of the session. Session 2 concentrated *sums*, and it is tempting to think averaging is what made it work. It is not. Ask what actually mattered: changing one $X_i$ shifts $\bar X$ by at most $(b-a)/n$ — the average is **stable**, insensitive to any single input. McDiarmid's insight is that *stability is the real requirement*, and averaging is just one way to get it.

**The bounded-differences condition, stated precisely.** If changing any single input moves $f(X_1,\dots,X_n)$ by at most $c_i$, then
$$P(|f - Ef| \ge t) \le 2\exp\!\left(\frac{-2t^2}{\sum_i c_i^2}\right)$$
— the same Gaussian tail as Hoeffding, for an *arbitrary* function. Not linear, not an average, not even continuous. Just insensitive to any one coordinate.

The longest-run statistic qualifies: flipping one coin can lengthen or shorten a run, but only locally, so a single flip moves the answer by a small bounded amount. Hence concentration, despite the statistic being a maximum over a combinatorial structure.

**Be clear about what this cell does and does not do.** It *demonstrates* that a non-average statistic concentrates. It does not compute an explicit $c_i$ and evaluate McDiarmid's bound against the empirical 0.023 — the code comment's "$c_i = \ldots$ small" is genuinely hand-waving, and a rigorous treatment would need to bound the effect of one flip carefully (it is not entirely obvious, since one flip can join two runs into a longer one). Read this as motivation for the theorem rather than a verification of it.

**And here is why the generalisation matters enough to spend a session on.** Empirical risk is an average of bounded losses, so swapping one training example moves it by at most $1/n$ — Hoeffding would have sufficed. But learning theory needs to control
$$\sup_{f \in \mathcal{F}} \big|\text{train risk}(f) - \text{true risk}(f)\big|,$$
a **supremum over an entire model class**, which is emphatically not an average and which Hoeffding cannot touch. McDiarmid can: swapping one example moves that supremum by at most $1/n$ as well, because it moves every individual term by at most that much.

That is the licence to talk about generalisation at all, and Session 4 is built entirely on it. Concentration for *stable functions* rather than merely for averages is what makes statistical learning theory possible.

---
### 🕐 Session 4 of 4 — *Rademacher Complexity & a Real Generalization Bound* (~40 min)
**Goal:** measure a model class's ability to fit NOISE; compute an honest bound for a trained model.
**Builds on:** Session 3.

---

<details>
<summary>🎓 <b>Teacher notes — Session 4: Rademacher Complexity & a Real Generalization Bound</b></summary>

**Timing (~40 min).** 8 min the question ML must answer · 12 min Rademacher complexity as noise-fitting · 10 min the two-class demo · 10 min cashing the bound.

**Board first — pose the question sharply.** My model fit the training data. Why should it fit *new* data? Nothing in the training procedure looked at new data, so the answer cannot come from the fit itself; it has to come from a property of the **model class**. That reframing — from "my model is good" to "my class is small enough" — is the conceptual core of statistical learning theory.

**Rademacher complexity, defined by what it measures.** $\mathcal{R}_n = E\sup_f \frac{1}{n}\sum_i \sigma_i f(x_i)$ with random signs: **the class's average ability to correlate with pure noise**. If a class can fit coin flips, then fitting your data proves nothing, because it would have fitted anything. If it cannot, a good fit is evidence. Say it that way before writing the formula — the definition then reads as an obvious operationalisation.

**The demo makes it a measurement, which is the unusual part.** Most treatments leave Rademacher complexity abstract; here it is *estimated by actually trying to fit random labels*. Linear classifiers score 0.124; the memorising class scores exactly 1.000 — it fits every random labelling perfectly, by construction. Ask what bound the second class earns: none. $2\mathcal{R}_n = 2$ makes the guarantee vacuous, and correctly so.

**Note the naming wart.** The function is called `stump_forest_fit` but its body is `return y` — pure memorisation, 1-NN evaluated on its own training set. The name is misleading; the comment corrects it. Worth saying aloud so students do not think a stump forest necessarily has $\mathcal{R} = 1$.

**Then cash the bound and be honest about looseness.** Train error 0.058, true error 0.070, bound 0.215. The bound *holds* — that is the point — and it is about 3× above the truth, with the gap term (0.157) more than ten times the real gap (0.012). Ask whether that makes it useless. No: it is **finite, computable, and assumption-explicit**, which is more than most heuristics offer. But it is not a prediction of test error, and using it as one would be a misuse.

**Ask the room.** "The bound says 0.215 and the truth is 0.070. Would you ship this model?" The bound is a worst-case guarantee over the whole class, not an estimate for the particular model you trained — so for deciding whether to ship, a held-out set is the right tool. Bounds tell you what is *guaranteed*; validation tells you what is *likely*. Both have uses and they are not interchangeable.

**Close on where this framework strains.** Modern deep networks have enormous capacity — they can fit random labels completely, so $\mathcal{R}_n \approx 1$ and these bounds go vacuous — and yet they generalise well. That is the [double descent](../../Intro_Mach_Learn/Training_Dynamics.ipynb) puzzle, and the value of this session is that students now know *exactly which step* fails: not the concentration, which is solid, but the uniform-over-the-class supremum, which is far too pessimistic for classes explored by gradient descent from a particular initialisation.
</details>

## 5. Why Learning Works (When It Does)

💡 **Intuition.** The question ML must answer: my model fit the *training* data — why should it fit *new* data? Answer: because the gap $\sup_{f \in \mathcal{F}} |\text{train risk} - \text{true risk}|$ concentrates (McDiarmid!) around its mean, and that mean is controlled by **Rademacher complexity**: $\mathcal{R}_n = E \sup_f \frac{1}{n} \sum_i \sigma_i f(x_i)$ with random signs $\sigma_i$ — literally *the class's average ability to correlate with pure noise*. Small class + can't fit coin flips ⇒ training performance transfers:
$$\text{true risk} \le \text{train risk} + 2\mathcal{R}_n + \sqrt{\ln(1/\delta)/2n}.$$

In [5]:
# Estimate Rademacher complexity of two REAL model classes by... trying to fit coin flips
import itertools

def rademacher_est(fit_predict, X, n_trials=60):
    vals = []
    for _ in range(n_trials):
        sigma = rng.choice([-1.0, 1.0], len(X))
        vals.append(np.mean(sigma * fit_predict(X, sigma)))
    return np.mean(vals)

X = rng.standard_normal((60, 2))

def linear_fit(X, y):                                   # linear classifiers (small class)
    w, *_ = np.linalg.lstsq(X, y, rcond=None)
    return np.sign(X @ w)

def stump_forest_fit(X, y):                             # 1-nearest-neighbor (huge class!)
    return y                                            # 1-NN on its own training set fits ANY labels

R_lin = rademacher_est(linear_fit, X)
R_1nn = rademacher_est(stump_forest_fit, X)
print(f"empirical Rademacher complexity, n=60:  linear {R_lin:.3f}   1-NN-style {R_1nn:.3f}")
print("→ the class that can memorize noise (R ≈ 1) earns NO generalization guarantee;")
print("  the linear class (R ≈ {:.2f} ≈ O(1/√n)) does.".format(R_lin))

empirical Rademacher complexity, n=60:  linear 0.124   1-NN-style 1.000
→ the class that can memorize noise (R ≈ 1) earns NO generalization guarantee;
  the linear class (R ≈ 0.12 ≈ O(1/√n)) does.


**What just happened.** Two model classes, and their ability to fit **pure noise** measured directly: linear classifiers score $\mathcal{R}_n = 0.124$; the memorising class scores exactly **1.000**.

**Rademacher complexity is defined as exactly this experiment.** $\mathcal{R}_n = E\sup_f \frac{1}{n}\sum_i \sigma_i f(x_i)$ with random signs $\sigma_i$ — the class's average correlation with random labels. So the "estimate" here is not an approximation to some abstract quantity; it is the quantity, computed by doing what the definition describes: generate coin flips, fit, measure correlation, repeat.

**The 1.000 is the important number.** A class that fits *every* random labelling perfectly has learned nothing when it fits yours. It would have fitted anything, so a good training score is zero evidence about new data. And the bound reflects that honestly: with $2\mathcal{R}_n = 2$, the guarantee
$$\text{true risk} \le \text{train risk} + 2\mathcal{R}_n + \sqrt{\ln(1/\delta)/2n}$$
becomes vacuous, since error rates cannot exceed 1 anyway. **No guarantee is the correct answer for a class that can memorise.**

The linear class's 0.124 is small and shrinking — roughly $O(\sqrt{d/n})$, which for $d = 2$, $n = 60$ predicts about 0.18, the right order. Small class, cannot fit noise, so a good fit transfers.

That is the reframing worth carrying: generalisation is not a property of your *model*, it is a property of your model *class*. Nothing about the training procedure looked at new data, so the guarantee has to come from the class being too restricted to fit arbitrary patterns.

**One naming wart to notice.** The function is called `stump_forest_fit`, but its body is `return y` — it simply returns whatever labels it was given, which is 1-NN evaluated on its own training set. The name is misleading and the comment corrects it; a genuine stump forest would have complexity between the two extremes shown here.

**And the honest position of this framework.** Modern deep networks can fit random labels completely, so $\mathcal{R}_n \approx 1$ and bounds of this form go vacuous — yet they generalise well in practice. That is the [double descent](../../Intro_Mach_Learn/Training_Dynamics.ipynb) puzzle. The value of having done this session carefully is knowing *which step* fails: not the concentration, which is solid, but the supremum taken uniformly over the whole class. Gradient descent from a particular initialisation explores a far smaller effective set than the class permits, and bounding the whole class is simply the wrong quantity.

In [6]:
# Cash the bound for a linear classifier on real (separable-ish) data
n_tr = 400
X_tr = rng.standard_normal((n_tr, 2)); y_tr = np.sign(X_tr @ np.array([1.5, -1.0]) + 0.4*rng.standard_normal(n_tr))
w, *_ = np.linalg.lstsq(X_tr, y_tr, rcond=None)
train_err = np.mean(np.sign(X_tr @ w) != y_tr)

R_n = rademacher_est(linear_fit, X_tr, 40)
delta = 0.05
bound = train_err + 2*abs(R_n) + np.sqrt(np.log(1/delta)/(2*n_tr))

X_te = rng.standard_normal((100_000, 2)); y_te = np.sign(X_te @ np.array([1.5, -1.0]) + 0.4*rng.standard_normal(100_000))
test_err = np.mean(np.sign(X_te @ w) != y_te)
print(f"train error {train_err:.3f}   TRUE error {test_err:.3f}   bound (95%) {bound:.3f}")
print("the bound holds — loose (bounds always are), but finite, honest, and assumption-explicit")

train error 0.058   TRUE error 0.070   bound (95%) 0.215
the bound holds — loose (bounds always are), but finite, honest, and assumption-explicit


**What just happened.** An actual generalisation bound, computed for a model that was actually trained:

| quantity | value |
|---|---|
| train error | 0.058 |
| **true** error (100,000 fresh points) | 0.070 |
| **95% bound** | **0.215** |

**The bound holds**, which is the claim being tested — and it is loose by about 3×. Worth being precise about *where* the looseness lives: the guaranteed gap is $0.215 - 0.058 = 0.157$, while the real gap is $0.070 - 0.058 = 0.012$. The bound over-estimates the generalisation gap by more than tenfold.

**Is it therefore useless? No — but it is not what people often want it to be.** It is **finite, computable from training data alone, and assumption-explicit**: it says exactly what it assumes (bounded loss, i.i.d. sampling, this model class) and delivers a guarantee that holds with probability 0.95 no matter what the data distribution is. Very few things in machine learning offer that.

What it is *not* is an estimate of test error. Using 0.215 as a prediction would be wrong, and the right tool for "how will this model do?" is a held-out set. **Bounds tell you what is guaranteed; validation tells you what is likely.** They answer different questions and are not interchangeable.

**Why the bound must be pessimistic.** It controls $\sup_{f\in\mathcal{F}}|\text{train} - \text{true}|$ — the worst case over the *entire class* — because the training procedure could in principle have returned any member of it. Your particular fitted $w$ is almost certainly not that worst case, so you pay for a possibility that did not occur. That is the price of a guarantee that holds without knowing which model you would end up with.

**The three terms are worth naming individually**, since each is a separate idea from this workshop:

- **train error** — what you measured.
- $2\mathcal{R}_n$ — the class's ability to fit noise (Session 4), and by far the dominant term here.
- $\sqrt{\ln(1/\delta)/2n}$ — the concentration term (Sessions 1–3), about 0.06 at $n = 400$ and $\delta = 0.05$.

Notice the concentration term is *small*. The looseness is almost entirely in the complexity term, which tells you where to invest: restricting the model class buys far more guarantee than collecting more data does, at this sample size.

**And this is the honest state of the theory.** For a linear model on 400 points the bound is loose but meaningful. For a modern deep network it goes fully vacuous, because such networks can fit random labels and so $\mathcal{R}_n \approx 1$ — yet they generalise. Knowing precisely which step breaks (the uniform supremum over the class, not the concentration) is what makes this session worth the time even where its conclusion no longer applies.

## 6. Conclusion

One trick (Markov on $e^{\lambda X}$) turns boundedness into Gaussian tails; independence adds proxies; stability replaces 'average'; and a class's ability to fit noise is precisely what generalization costs. The [double-descent mystery](../../Intro_Mach_Learn/Training_Dynamics.ipynb) is the modern frontier where these classical bounds strain — now you know exactly *which* step strains.

---
## Where next

- [Stochastic Processes II](../Stochastic_Processes/Stochastic_Processes_2.ipynb) — the martingale machinery under McDiarmid.
- [Online Learning](../../Intro_Time_Series/Online_Learning_and_Regret.ipynb) — the assumption-free sibling theory.
- [Random Matrix Theory](../Random_Matrix_Theory/Random_Matrix_Theory.ipynb) — concentration for eigenvalues.